In [10]:
__author__ = 'Nico Matthew Valencia'

# This file runs summary-based mendelian randomization using GTEX files to Khat Use and Khat Use
# Frequency summary statistics

In [11]:
import hailtop.batch as hb

gwas_files = {
    "ku": "gs://neurogap-bge-imputed-regional/nico/khat_gwas/meta_analysis_outputs/gp08/adjusted/gwaspy/corrected/studysite_covar_all/ku_gp08_meta_cleaned_with_N.smr.tsv",
    "kuf": "gs://neurogap-bge-imputed-regional/nico/khat_gwas/meta_analysis_outputs/gp08/adjusted/gwaspy/corrected/studysite_covar_all/kuf_gp08_meta_cleaned_with_N.smr.tsv",
}


GTEX_V10_DIR = "gs://neurogap-bge-imputed-regional/nico/khat_gwas/smr/data/gtex_v10"

# Only ran SMR on whichever tissues were significant for either KU/KUF
gtex_tissues = [
    #"Adipose_Subcutaneous",
    #"Adipose_Visceral_Omentum",
    #"Adrenal_Gland",
    "Artery_Aorta",
    #"Artery_Coronary",
    "Artery_Tibial",
    #"Brain_Amygdala",
    #"Brain_Anterior_cingulate_cortex_BA24",
    #"Brain_Caudate_basal_ganglia",
    #"Brain_Cerebellar_Hemisphere",
    "Brain_Cerebellum",
    #"Brain_Cortex",
    #"Brain_Frontal_Cortex_BA9",
    #"Brain_Hippocampus",
    #"Brain_Hypothalamus",
    #"Brain_Nucleus_accumbens_basal_ganglia",
    #"Brain_Putamen_basal_ganglia",
    #"Brain_Spinal_cord_cervical_c-1",
    #"Brain_Substantia_nigra",
    #"Breast_Mammary_Tissue",
    "Cells_Cultured_fibroblasts",
    #"Cells_EBV-transformed_lymphocytes",
    #"Colon_Sigmoid",
    #"Colon_Transverse",
    #"Esophagus_Gastroesophageal_Junction",
    #"Esophagus_Mucosa",
    #"Esophagus_Muscularis",
    #"Heart_Atrial_Appendage",
    #"Heart_Left_Ventricle",
    #"Kidney_Cortex",
    "Liver",
    #"Lung",
    #"Minor_Salivary_Gland",
    "Muscle_Skeletal",
    "Nerve_Tibial",
    #"Ovary",
    #"Pancreas",
    #"Pituitary",
    "Prostate",
    "Skin_Not_Sun_Exposed_Suprapubic",
    #"Skin_Sun_Exposed_Lower_leg",
    #"Small_Intestine_Terminal_Ileum",
    #"Spleen",
    #"Stomach",
    "Testis",
    "Thyroid",
    #"Uterus",
    #"Vagina",
    "Whole_Blood",
]

CHRS = list(range(1, 23))

gtex_prefixes = {
    tissue: {
        chrom: f"{GTEX_V10_DIR}/{tissue}.v10.eQTL.cis_qtl_pairs.{chrom}"
        for chrom in CHRS
    }
    for tissue in gtex_tissues
}

bfile_paths = {
    "bed": "gs://neurogap-bge-imputed-regional/nico/khat_gwas/reference_bfile/no_related_SNPdups/ngap_subsetted/hgdp_1kgp_AFR_unrelated_nodup_subset_to_neurogap.bed",
    "bim": "gs://neurogap-bge-imputed-regional/nico/khat_gwas/reference_bfile/no_related_SNPdups/ngap_subsetted/hgdp_1kgp_AFR_unrelated_nodup_subset_to_neurogap.bim",
    "fam": "gs://neurogap-bge-imputed-regional/nico/khat_gwas/reference_bfile/no_related_SNPdups/ngap_subsetted/hgdp_1kgp_AFR_unrelated_nodup_subset_to_neurogap.fam",
}


JOB_REGIONS = None
DEFAULT_THREADS = 2
DEFAULT_MEMORY = "128Gi"
DEFAULT_STORAGE = "300Gi"
# default threads is 1, but setting to 10 to make it hopefully run faster

print("n gwas files:", len(gwas_files))
print("n tissues:", len(gtex_prefixes))
print("threads per job:", DEFAULT_THREADS)
print("memory per job:", DEFAULT_MEMORY)
print("storage per job:", DEFAULT_STORAGE)

n gwas files: 2
n tissues: 12
threads per job: 2
memory per job: 128Gi
storage per job: 300Gi


In [12]:
def run_smr_job(
    b: hb.Batch,
    gwas_label: str,
    gwas_summary_file,
    bfile,
    beqtl_rg,
    tissue_name: str,
    chrom: int,
    threads: int = DEFAULT_THREADS,
    memory: str = DEFAULT_MEMORY,
    storage: str = DEFAULT_STORAGE,
    regions=JOB_REGIONS,
):
    j = b.new_job(name=f"SMR {gwas_label} x {tissue_name} chr{chrom}")
    j.image("nsvalencia/smr:v1")
    j.cpu(threads)
    j.memory(memory)
    j.storage(storage)

    if regions is not None:
        j.regions(regions)

    j.declare_resource_group(
        smr_out={
            "smr": "{root}.smr",
            "log": "{root}.log",
            "status": "{root}.status.txt",
        }
    )

    j.command(
        f"""
set -euxo pipefail

echo "=== staged inputs ==="
ls -lah

echo "=== standardizing PLINK reference prefix ==="
ln -s {bfile.bed} geno.bed
awk 'BEGIN{{OFS="\\t"}} NR>1 {{sub(/^chr/, "", $1); print}}' {bfile.bim} > geno.bim
ln -s {bfile.fam} geno.fam

echo "=== standardizing eQTL prefix ==="
ln -s {beqtl_rg.besd} eqtl.besd
ln -s {beqtl_rg.epi} eqtl.epi
ln -s {beqtl_rg.esi} eqtl.esi

echo "=== after linking inputs ==="
ls -lah

echo "=== checking PLINK files ==="
wc -l geno.bim geno.fam
head geno.bim
head geno.fam

echo "=== checking eQTL files ==="
wc -l eqtl.esi eqtl.epi
head eqtl.esi
head eqtl.epi

echo "=== checking GWAS file ==="
head {gwas_summary_file}

set +e
smr \\
  --bfile geno \\
  --gwas-summary {gwas_summary_file} \\
  --beqtl-summary eqtl \\
  --diff-freq 1 \\
  --heidi-mtd 1 \\
  --thread-num {threads} \\
  --out smr_result \\
  > {j.smr_out.log} 2>&1
smr_exit=$?
set -e

echo "SMR exit code: $smr_exit" > {j.smr_out.status}
echo "" >> {j.smr_out.status}

if [ -s smr_result.smr ]; then
  echo "SMR output file exists" >> {j.smr_out.status}
  mv smr_result.smr {j.smr_out.smr}
else
  echo "SMR output file missing" >> {j.smr_out.status}
  echo "No smr_result.smr was produced." >> {j.smr_out.status}
  touch {j.smr_out.smr}
fi

echo "" >> {j.smr_out.status}
echo "=== working directory after SMR ===" >> {j.smr_out.status}
ls -lah >> {j.smr_out.status} 2>&1

exit $smr_exit
"""
    )

    return j

In [15]:
# specific CHR & tissue rerun of either KU or KUF --> also test
backend = hb.ServiceBackend(
    billing_project="neale-pumas-bge",
    remote_tmpdir="gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp",
)

b = hb.Batch(
    backend=backend,
    name="SMR Rerun: Aortic Artery; KU; chr2",
)

bfile = b.read_input_group(
    bed=bfile_paths["bed"],
    bim=bfile_paths["bim"],
    fam=bfile_paths["fam"],
)

gwas_label = "ku"
gwas_summary_file = b.read_input(gwas_files[gwas_label])

tissue_name = "Artery_Aorta"
chrom = 2

tissue_prefix = gtex_prefixes[tissue_name][chrom]

beqtl_rg = b.read_input_group(
    besd=f"{tissue_prefix}.besd",
    epi=f"{tissue_prefix}.epi",
    esi=f"{tissue_prefix}.esi",
)

job = run_smr_job(
    b=b,
    gwas_label=gwas_label,
    gwas_summary_file=gwas_summary_file,
    bfile=bfile,
    beqtl_rg=beqtl_rg,
    tissue_name=tissue_name,
    chrom=chrom,
    threads=4,
    memory="32Gi",
    storage="300Gi",
)

out_base = (
    f"gs://neurogap-bge-imputed-regional/nico/khat_gwas/smr/output/studysite_covar_all"
    f"{gwas_label}_gtexv10_test/{tissue_name}/chr{chrom}/"
    f"{gwas_label}_{tissue_name}_chr{chrom}"
)

b.write_output(job.smr_out, out_base)

print("jobs in test batch:", len(b._jobs))

b.run(wait=False)
backend.close()

jobs in test batch: 1


In [14]:
# run ALL tissues
backend = hb.ServiceBackend(
    billing_project="neale-pumas-bge",
    remote_tmpdir="gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp",
)

b = hb.Batch(
    backend=backend,
    name="SMR: KU & KUF X GTExV10 all tissues by chromosome",
)

bfile = b.read_input_group(
    bed=bfile_paths["bed"],
    bim=bfile_paths["bim"],
    fam=bfile_paths["fam"],
)

for gwas_label, gwas_path in gwas_files.items():
    gwas_summary_file = b.read_input(gwas_path)

    for tissue_name, chr_prefixes in gtex_prefixes.items():
        for chrom, tissue_prefix in chr_prefixes.items():

            beqtl_rg = b.read_input_group(
                besd=f"{tissue_prefix}.besd",
                epi=f"{tissue_prefix}.epi",
                esi=f"{tissue_prefix}.esi",
            )

            job = run_smr_job(
                b=b,
                gwas_label=gwas_label,
                gwas_summary_file=gwas_summary_file,
                bfile=bfile,
                beqtl_rg=beqtl_rg,
                tissue_name=tissue_name,
                chrom=chrom,
                threads=4,
                memory="32Gi",
                storage="300Gi",
            )

            out_base = (
                f"gs://neurogap-bge-imputed-regional/nico/khat_gwas/smr/output/studysite_covar_all"
                f"{gwas_label}_gtexv10/{tissue_name}/chr{chrom}/"
                f"{gwas_label}_{tissue_name}_chr{chrom}"
            )

            b.write_output(job.smr_out, out_base)

print("jobs in production batch:", len(b._jobs))

b.run(wait=False)
backend.close()

jobs in production batch: 528


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/rich/live.py:229: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [ ]:
# Run this only if you need to rerun a specific tissue
TISSUE_TO_RERUN = "Artery_Aorta"

backend = hb.ServiceBackend(
    billing_project="neale-pumas-bge",
    remote_tmpdir="gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp",
)

b = hb.Batch(
    backend=backend,
    name=f"SMR RERUN: KU & KUF X GTExV10 {TISSUE_TO_RERUN}",
)

bfile = b.read_input_group(
    bed=bfile_paths["bed"],
    bim=bfile_paths["bim"],
    fam=bfile_paths["fam"],
)

for gwas_label, gwas_path in gwas_files.items():
    gwas_summary_file = b.read_input(gwas_path)

    for chrom, tissue_prefix in gtex_prefixes[TISSUE_TO_RERUN].items():

        beqtl_rg = b.read_input_group(
            besd=f"{tissue_prefix}.besd",
            epi=f"{tissue_prefix}.epi",
            esi=f"{tissue_prefix}.esi",
        )

        job = run_smr_job(
            b=b,
            gwas_label=gwas_label,
            gwas_summary_file=gwas_summary_file,
            bfile=bfile,
            beqtl_rg=beqtl_rg,
            tissue_name=TISSUE_TO_RERUN,
            chrom=chrom,
            threads=4,
            memory="32Gi",
            storage="300Gi",
        )

        out_base = (
            f"gs://neurogap-bge-imputed-regional/nico/khat_gwas/smr/output/studysite_covar_all"
            f"{gwas_label}_gtexv10/{TISSUE_TO_RERUN}/chr{chrom}/"
            f"{gwas_label}_{TISSUE_TO_RERUN}_chr{chrom}"
        )

        b.write_output(job.smr_out, out_base)

print("jobs in tissue rerun batch:", len(b._jobs))

b.run(wait=False)
backend.close()